# How a Computer Works

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Say what the CPU, RAM and storage do, and why a program is copied into RAM to run
- Trace the fetch-decode-execute cycle of a short program
- Explain why caches make programs faster, and pick the access pattern that uses them well
- Describe how javac and the JVM turn Java code into instructions the CPU runs

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## In short

A computer has four main parts. The **CPU** runs instructions; **RAM** holds the programs and data in use right now, and forgets everything when the power goes off; **storage** (an SSD or a disk) keeps files; **input and output** devices talk to the world. To run, a program is copied from storage into RAM, because storage is far too slow for the CPU to read instructions from.

The CPU repeats one cycle billions of times per second: it **fetches** the instruction whose address is in the **program counter**, **decodes** what it asks for, **executes** it, and moves on to the next one, unless the instruction was a **jump**, which is how `if` and loops work at this level. A 3 GHz clock ticks 3 billion times per second.

![The fetch-decode-execute cycle](cycle.svg)

#### 🎞️ The fetch-decode-execute cycle

![A three-instruction program run step by step: the program counter moves on and the accumulator goes from 0 to 5 to 8](cpu-cycle-steps.svg)

### ✏️ Exercise: A three-instruction CPU

An accumulator starts at 0. Run each instruction in order: `ADD n`, `SUB n` or `MUL n`, and return the accumulator at the end. That loop is the fetch-decode-execute cycle.

In [ ]:
int run(String[] program) {
    // TODO: split each instruction into its name and number, and apply it
    return 0;
}

void main() {
    check("add and multiply", run(new String[] {"ADD 5", "MUL 3"}), 15);
    check("subtract", run(new String[] {"ADD 10", "SUB 4", "SUB 7"}), -1);
    check("no program", run(new String[] {}), 0);
}

<details>
<summary>Show an answer</summary>

```java
int run(String[] program) {
    int acc = 0;
    for (String instruction : program) {
        String[] parts = instruction.split(" ");
        int n = Integer.parseInt(parts[1]);
        switch (parts[0]) {
            case "ADD" -> acc += n;
            case "SUB" -> acc -= n;
            case "MUL" -> acc *= n;
            default -> throw new IllegalArgumentException(instruction);
        }
    }
    return acc;
}

void main() {
    check("add and multiply", run(new String[] {"ADD 5", "MUL 3"}), 15);
    check("subtract", run(new String[] {"ADD 10", "SUB 4", "SUB 7"}), -1);
    check("no program", run(new String[] {}), 0);
}
```

</details>

RAM is about 100 times slower than the CPU, so processors keep recently used data in small, fast **caches**. Data already in the cache (a **hit**) arrives in about 1 ns; a **miss** waits about 100 ns for RAM. A miss brings a whole block of neighbouring bytes, so reading an array in order is much faster than jumping around in it.

**Try it:** the same 16 million numbers, added row by row and then column by column. Row by row reads memory in order, so most reads are cache hits.

In [ ]:
int n = 4000;
int[][] grid = new int[n][n];
for (int[] row : grid) {
    Arrays.fill(row, 1);
}

long start = System.nanoTime();
long byRows = 0;
for (int r = 0; r < n; r++) {
    for (int c = 0; c < n; c++) {
        byRows += grid[r][c];
    }
}
long middle = System.nanoTime();
long byColumns = 0;
for (int c = 0; c < n; c++) {
    for (int r = 0; r < n; r++) {
        byColumns += grid[r][c];
    }
}
long end = System.nanoTime();

IO.println("row by row:       " + (middle - start) / 1_000_000 + " ms (sum " + byRows + ")");
IO.println("column by column: " + (end - middle) / 1_000_000 + " ms (sum " + byColumns + ")");

### ✏️ Exercise: Same cache block?

A cache moves memory in blocks: with blocks of 64 bytes, addresses 0 to 63 are block 0, 64 to 127 block 1, and so on. Do two addresses fall in the same block?

In [ ]:
boolean sameBlock(int a, int b, int blockSize) {
    // TODO: compare the block numbers of a and b
    return false;
}

void main() {
    check("0 and 63", sameBlock(0, 63, 64), true);
    check("63 and 64", sameBlock(63, 64, 64), false);
    check("130 and 140", sameBlock(130, 140, 16), true);
}

<details>
<summary>Show an answer</summary>

```java
boolean sameBlock(int a, int b, int blockSize) {
    return a / blockSize == b / blockSize;
}

void main() {
    check("0 and 63", sameBlock(0, 63, 64), true);
    check("63 and 64", sameBlock(63, 64, 64), false);
    check("130 and 140", sameBlock(130, 140, 16), true);
}
```

</details>

Java adds one step: `javac` compiles your code into **bytecode**, and the **JVM** runs it, turning the code that runs most into real machine code for your CPU.

**Your computer, as Java sees it:**

In [ ]:
Runtime runtime = Runtime.getRuntime();
IO.println("Java version:     " + System.getProperty("java.version"));
IO.println("Operating system: " + System.getProperty("os.name") + " (" + System.getProperty("os.arch") + ")");
IO.println("Processor cores:  " + runtime.availableProcessors());
IO.println("Memory the JVM may use: " + runtime.maxMemory() / (1024 * 1024) + " MiB");

## Readings

- [The Central Processing Unit (CPU)](https://www.youtube.com/watch?v=FZGugFqdr60) (Crash Course Computer Science #7, 10 min): Follow one instruction through fetch, decode and execute. Notice the instruction address register (the program counter) and how the clock drives each step.
- [Advanced CPU Designs](https://www.youtube.com/watch?v=rtAlC5J1U40) (Crash Course Computer Science #9, 12 min): Why RAM is slow next to the CPU, how a cache keeps whole blocks of data close, and what a cache hit and a miss are. Pipelining and multiple cores are extras.
- [How JVM Works: JVM Architecture](https://www.geeksforgeeks.org/java/how-jvm-works-jvm-architecture/) (GeeksforGeeks, 8 min): What javac produces, how the class loader brings .class files into memory, and how the JIT compiler turns code that runs often into machine code.

## Check yourself

1. What does the program counter hold, and when does it not simply move to the next instruction?
2. Why can reading a big array in order be ten times faster than reading it in a random order?
3. What does `javac` produce, and what turns it into instructions your CPU can run?

The quiz checks these, and the challenges let you build a small CPU, a stack machine and a cache.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Instruction Decoder** · Easy
- **Stack Machine** · Medium
- **Cache Simulator** · Medium
- **Tiny CPU** · Hard